In [3]:
# Imports

import torch
import torch.nn as nn
import torchvision.models as tvmodels
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms 

import numpy as np
import rasterio

import os

from skimage.io import imread

In [4]:
#TOTAL NUMBER OF IMAGES ARE AROUND 21000

In [ ]:
pre_datafolder= ''
post_datafolder = ''
    
pre_pathlist = [os.path.join(pre_datafolder,img.endswith(".tif")) for img in os.listdir(pre_datafolder)]

post_pathlist= [os.path.join(post_datafolder,img.endswith(".tif")) for img in os.listdir(post_datafolder)]

In [ ]:
pre_meanstd = {"mean":[2681.30,2673.31,2675.80],"std":[12729.79,12731.37,12730.86]}
post_meanstd = {"mean":[],"std":[]}

In [ ]:
# Dataset and Dataloader

class CustomDataset(Dataset):

    def __init__(self,pre,post):
        super().__init__()

        self.pre = pre
        self.post = post


    def __getitem__(self, index):
        pre = self.pre[index]
        post = self.post[index]

        pre = imread(pre)
        post = imread(post)

        return pre, post
    
    def __len__(self):
        return len(self.pre)
    

predata_transformation = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(pre_meanstd["mean"],pre_meanstd["std"]),
        transforms.ConvertImageDtype(torch.float)
    ])
    
dataset = CustomDataset(pre_pathlist,post_pathlist)

train_loader = DataLoader(dataset,shuffle=True,batch_size=200,transforms=predata_transformation)

In [12]:
# Model

# Pre trained VIT encoder

class VITencoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = tvmodels.vit_b_16(weights=True)

        self.encoder.heads = nn.Identity()


    def forward (self,x):

        x = self.encoder(x)

        return x
    
#Linear lightweight FCN layer decoder

class LightDecoderM(nn.Module):

    def __init__(self, ):
        super().__init__()

        self.decoder = nn.Sequential(
            nn.Conv2d(
                in_channels=768,
                out_channels=self.encoder_stride ** 2 * 3, kernel_size=1),
            nn.PixelShuffle(self.encoder_stride),
        )

        self.in_chans = self.encoder.in_chans
        self.patch_size = self.encoder.patch_size

    def forward(self,x):

        x = self.decoder(x)

        return x


In [ ]:
# Build Sharedweight Encoder

In [ ]:
# Training Pipeline
